### Load data first

In [1]:
import os
import numpy as np
import time
import sys

import pandas as pd
from tqdm import tqdm
from datetime import datetime
from dataclasses import dataclass
from typing import Literal, Annotated


sys.path.append(os.path.abspath(".."))   # Add root path to sys.path
os.chdir("..")  # Change working directory to root path

from src.models import BasePrecipitationModel

DATASET_PATH = "data/numpy_grid/"
RESULT_PATH = "data/tracking_output/"


## You are using the Python ARM Radar Toolkit (Py-ART), an open source
## library for working with weather radar data. Py-ART is partly
## supported by the U.S. Department of Energy as part of the Atmospheric
## Radiation Measurement (ARM) Climate Research Facility, an Office of
## Science user facility.
##
## If you use this software to prepare a publication, please cite:
##
##     JJ Helmus and SM Collis, JORS 2016, doi: 10.5334/jors.119



### Setup Parameters

In [ ]:

@dataclass
class ModelSetting:
    identification_method: Annotated[str, "Method for identifying objects"]
    model_name: Annotated[str, "Name of the model"]
    model: BasePrecipitationModel
    maximum_velocity: int

@dataclass
class SensitivityAnalysisResults:
    pod_3: float
    far_3: float
    csi_3: float
    pod_5: float
    far_5: float
    csi_5: float
    object_consistency: float
    mean_duration: float
    linear_rmse: float
    optimal_tracking: float

def check_override_result(dataset: str, analysis: ModelSetting) -> pd.DataFrame:
    """
        Implement logic to check if results already exist for the given dataset and analysis parameters
    """
    saved_path = os.path.join(RESULT_PATH, f"{dataset}_models_evaluation.csv")
    if os.path.exists(saved_path):
        df = pd.read_csv(saved_path)
    else:
        df = pd.DataFrame(columns=["model_name", "maximum_velocity", "pod_3", "far_3", "csi_3", "pod_5", "far_5", "csi_5",
                                   "object_consistency", "mean_duration", "linear_rmse", "optimal_tracking"])
    
    # Check if the specific analysis parameters already exist in the dataframe
    existing = df[
        (df["model_name"] == analysis.model_name) &
        (df["maximum_velocity"] == analysis.maximum_velocity)
    ]

    return existing


def save_results(dataset: str, analysis: ModelSetting, results: SensitivityAnalysisResults):
    # Implement logic to save the results to disk
    saved_path = os.path.join(RESULT_PATH, f"{dataset}_models_evaluation.csv")
    if os.path.exists(saved_path):
        df = pd.read_csv(saved_path)
    else:
        df = pd.DataFrame(columns=["model_name", "maximum_velocity", "pod_3", "far_3", "csi_3", "pod_5", "far_5", "csi_5",
                                   "object_consistency", "mean_duration", "linear_rmse", "optimal_tracking"])

    new_df = pd.Series({
        "model_name": analysis.model_name,
        "maximum_velocity": analysis.maximum_velocity,
        "pod_3": results.pod_3,
        "far_3": results.far_3,
        "csi_3": results.csi_3,
        "pod_5": results.pod_5,
        "far_5": results.far_5,
        "csi_5": results.csi_5,
        "object_consistency": results.object_consistency,
        "mean_duration": results.mean_duration,
        "linear_rmse": results.linear_rmse,
        "optimal_tracking": results.optimal_tracking
    })

    # Override the existing rows if they exist
    existing = check_override_result(dataset, analysis)
    if existing is not None and not existing.empty:
        exiting_idx = existing.index[0]
        df.loc[exiting_idx] = new_df
    else:
        df.loc[len(df)] = new_df
        
    df.to_csv(saved_path, index=False)

In [3]:
def run_models_evaluation(dataset: str, analysis: ModelSetting) -> SensitivityAnalysisResults:
    """
        Main entries to run sensitivity analysis. Phases include:
            1. Load dataset and configure model with given parameters
            2. Identify storms in each frame
            3. Perform and evaluate nowcasting across time
            4. Evaluation of tracking performance
    """
    # Phase 1: Load dataset and configure model
    from src.preprocessing import read_numpy_grid

    source_path = os.path.join(DATASET_PATH, dataset)
    img_paths = [os.path.join(source_path, img_name) for img_name in sorted(os.listdir(source_path)) if img_name.endswith('.npy')]
    dbz_maps: list[tuple[np.ndarray, datetime]] = []

    for path in tqdm(img_paths, desc="Processing images and detecting storms"):
        file_name = path.split("/")[-1].split(".")[0]

        time_frame = datetime.strptime(file_name[4:19], "%Y%m%d_%H%M%S")
        img = read_numpy_grid(path)
        dbz_maps.append((img, time_frame))

    # Phase 2: Identify storms
    storms_maps = []
    for idx, (dbz_map, time_frame) in tqdm(list(enumerate(dbz_maps)), desc="Identifying storms in each frame"):
        storms_map = analysis.model.identify_storms(dbz_map, time_frame, map_id=f"time_{idx}", threshold=35, filter_area=50)
        storms_maps.append(storms_map)

    # Phase 3: Evaluate Nowcasting across time
    from src.cores.metrics import PredictionBenchmarkModel

    # Evaluate for 3-step ahead prediction
    ours_model_evaluation_3 = PredictionBenchmarkModel()
    temp_storm_map = storms_maps

    model = analysis.model.copy()
    SLOW_START_STEPS = 3
    PREDICT_FORWARD_STEPS = 3

    for i in range(SLOW_START_STEPS):
        model.processing_map(temp_storm_map[i])  # Warm-up phase

    for curr_map, future_map in tqdm(list(zip(temp_storm_map[SLOW_START_STEPS:], temp_storm_map[PREDICT_FORWARD_STEPS + SLOW_START_STEPS:])), desc="Predicting precipitation maps"):
        # Predict map using current data
        dt_seconds = (future_map.time_frame - model.storms_maps[-1].time_frame).total_seconds()
        predicted_map = model.forecast(dt_seconds)
        ours_model_evaluation_3.evaluate_predict(future_map, predicted_map)
        model.processing_map(curr_map)  # Update model with the current map

    # Evaluate for 5-step ahead prediction
    ours_model_evaluation_5 = PredictionBenchmarkModel()
    temp_storm_map = storms_maps

    model = analysis.model.copy()
    SLOW_START_STEPS = 5
    PREDICT_FORWARD_STEPS = 5

    for i in range(SLOW_START_STEPS):
        model.processing_map(temp_storm_map[i])  # Warm-up phase

    for curr_map, future_map in tqdm(list(zip(temp_storm_map[SLOW_START_STEPS:], temp_storm_map[PREDICT_FORWARD_STEPS + SLOW_START_STEPS:])), desc="Predicting precipitation maps"):
        # Predict map using current data
        dt_seconds = (future_map.time_frame - model.storms_maps[-1].time_frame).total_seconds()
        predicted_map = model.forecast(dt_seconds)
        ours_model_evaluation_5.evaluate_predict(future_map, predicted_map)
        model.processing_map(curr_map)  # Update model with the current map

    # Phase 4: Evaluate tracking performance
    from src.cores.metrics import PostEventClustering, linear_tracking_error

    # 4.1 Object consistency first
    object_consistency_scores = []
    for track in model.tracker.tracks:
        areas = [storm.contour.area for storm in track.storms.values()]
        area_changes = [abs(areas[i] - areas[i - 1]) / areas[i - 1] for i in range(1, len(areas)) if areas[i - 1] != 0]
        object_consistency_scores.append(np.mean(area_changes) if area_changes else 0)

    object_consistency_score = np.mean(object_consistency_scores)

    # 4.2 Mean duration of tracked objects
    mean_duration = np.mean([len(track.storms) for track in model.tracker.tracks])

    # 4.3 Linear RMSE
    linear_error_lsts = [linear_tracking_error(storm.history_movements[:-1]) ** 2 for storms_map in storms_maps 
                                                                    for storm in storms_map.storms if len(storm.history_movements) >= mean_duration]
    linear_rmse = np.sqrt(np.mean(linear_error_lsts))

    # 4.4 Optimal tracking evaluation
    centroids = []
    clusters_assigned = []

    FIRST_TIME_FRAME = dbz_maps[0][1]

    def _convert_time_frame_to_seconds(time_frame: datetime) -> float:
        return time_frame.timestamp() - FIRST_TIME_FRAME.timestamp()

    for track_history in model.tracker.tracks:
        track_centroids = [(storm.centroid[0], storm.centroid[1], _convert_time_frame_to_seconds(time_frame)) for time_frame, storm in track_history.storms.items()]
        centroids.extend(track_centroids)
        clusters_assigned.extend([track_history.id] * len(track_centroids))
    
    centroids = np.array(centroids)

    postevent_analysis = PostEventClustering(centroids, max_window_time=600, spatial_distance_threshold=50)
    reassigned_clusters_centers = postevent_analysis.fit_transform(num_clusters=len(model.tracker.tracks), clusters_assigned=clusters_assigned, max_epochs=50)

    merged_clusters_assigned = [postevent_analysis.clusters_merged_dict.get(i, i) for i in clusters_assigned]
    score_lst = [1 if merged_clusters_assigned[i] == reassigned_clusters_centers[i] else 0.5 if reassigned_clusters_centers[i] != -1 else 0 for i in range(len(clusters_assigned))]

    optimal_tracking_score = np.mean(score_lst)

    return SensitivityAnalysisResults(
        pod_3=np.mean(ours_model_evaluation_3.pods),
        far_3=np.mean(ours_model_evaluation_3.fars),
        csi_3=np.mean(ours_model_evaluation_3.csis),
        pod_5=np.mean(ours_model_evaluation_5.pods),
        far_5=np.mean(ours_model_evaluation_5.fars),
        csi_5=np.mean(ours_model_evaluation_5.csis),
        object_consistency=object_consistency_score,
        mean_duration=mean_duration,
        linear_rmse=linear_rmse,
        optimal_tracking=optimal_tracking_score
    )

## Running the test

In [18]:
from src.models import TitanPrecipitationModel, ETitanPrecipitationModel, STitanPrecipitationModel, \
                        ISCITPrecipitationModel, AdaptiveTrackingPrecipitationModel, OursPrecipitationModel
from src.identification import MorphContourIdentifier
from src.cores.movement_estimate import TREC

MAXIMUM_VELOCITY = 200
datasets = ["KARX", "KDVN", "KGRR"]
identifier = MorphContourIdentifier()

trec = TREC(max_velocity=MAXIMUM_VELOCITY, stride=32, block_size=32)

models = [
    TitanPrecipitationModel(identifier=identifier, max_velocity=MAXIMUM_VELOCITY),
    ETitanPrecipitationModel(identifier=identifier, trec=trec, max_velocity=MAXIMUM_VELOCITY),
    STitanPrecipitationModel(identifier=identifier, trec=trec, max_velocity=MAXIMUM_VELOCITY),
    ISCITPrecipitationModel(identifier=identifier, trec=trec, max_velocity=MAXIMUM_VELOCITY),
    AdaptiveTrackingPrecipitationModel(identifier=identifier, max_velocity=MAXIMUM_VELOCITY),
    OursPrecipitationModel(identifier=identifier, max_velocity=MAXIMUM_VELOCITY),
]
model_names = [
    "titan", 
    "etitan", "stitan", 
    "iscit", 
    "ata", "ours"
]

combinations = [
    (dataset, 
    ModelSetting(
        identification_method="morph_contour",
        model_name=model_name,
        model=model,
        maximum_velocity=MAXIMUM_VELOCITY
    )) for dataset in datasets for model, model_name in zip(models, model_names)
]

In [19]:
results = []

for idx, params in enumerate(combinations):
    print("-" * 50)
    dataset = params[0]
    param = params[1]

    print(f"Running combination {idx + 1}/{len(combinations)}: {param.model_name} on dataset {dataset}")

    if not check_override_result(dataset, param).empty:
        print("Skipping existing result.")
        continue

    results.append(run_models_evaluation(dataset, param))
    save_results(dataset, param, results[-1])

--------------------------------------------------
Running combination 1/18: titan on dataset KARX
Skipping existing result.
--------------------------------------------------
Running combination 2/18: etitan on dataset KARX
Skipping existing result.
--------------------------------------------------
Running combination 3/18: stitan on dataset KARX
Skipping existing result.
--------------------------------------------------
Running combination 4/18: iscit on dataset KARX
Skipping existing result.
--------------------------------------------------
Running combination 5/18: ata on dataset KARX
Skipping existing result.
--------------------------------------------------
Running combination 6/18: ours on dataset KARX
Skipping existing result.
--------------------------------------------------
Running combination 7/18: titan on dataset KDVN
Skipping existing result.
--------------------------------------------------
Running combination 8/18: etitan on dataset KDVN
Skipping existing result.


Predicting precipitation maps: 100%|██████████| 68/68 [00:00<00:00, 208.64it/s]


Post-event clustering:   0%|          | 0/50 [00:00<?, ?it/s]

--------------------------------------------------
Running combination 13/18: titan on dataset KGRR
Skipping existing result.
--------------------------------------------------
Running combination 14/18: etitan on dataset KGRR
Skipping existing result.
--------------------------------------------------
Running combination 15/18: stitan on dataset KGRR
Skipping existing result.
--------------------------------------------------
Running combination 16/18: iscit on dataset KGRR
Skipping existing result.
--------------------------------------------------
Running combination 17/18: ata on dataset KGRR
Skipping existing result.
--------------------------------------------------
Running combination 18/18: ours on dataset KGRR
Skipping existing result.


## Load model